# CIFAR-2 / ResNet-9 — LDS Benchmark

Compares **Traceprop** vs **TRAK (per-sample)** vs **Random baseline** on a real vision task.

Runs on CPU (Kaggle P100 is incompatible with current PyTorch).
Uses 20 retraining subsets and 500 test points for LDS to fit within Kaggle timeout.
Results saved to `cifar2_resnet9_lds.json`.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "dattri"])

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
import torchvision
import torchvision.transforms as transforms
from scipy.stats import spearmanr
import json
import time

# Force CPU to avoid P100/CUDA compatibility issues on Kaggle
DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

# --- Config ---
PROJ_DIM = 4096
N_SUBSETS = 20          # Reduced from 50 to fit CPU timeout
SUBSET_RATIO = 0.5
LR = 0.1
EPOCHS = 20
BATCH_SIZE = 128
SEED = 42
N_TEST_LDS = 500        # Evaluate LDS on 500 test points (not all 2000)

np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. CIFAR-2 Dataset (airplane vs automobile)

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

full_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
full_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Filter to classes 0 (airplane) and 1 (automobile)
def filter_cifar2(dataset):
    indices = [i for i, (_, label) in enumerate(dataset) if label in (0, 1)]
    return indices

# For ground-truth retraining we need raw (no augmentation) training data
raw_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform_test)

train_indices = [i for i in range(len(full_train)) if full_train.targets[i] in (0, 1)]
test_indices = [i for i in range(len(full_test)) if full_test.targets[i] in (0, 1)]

# Use raw (no augmentation) for deterministic retraining
train_subset = Subset(raw_train, train_indices)
test_subset = Subset(full_test, test_indices)

N_TRAIN = len(train_indices)
N_TEST = len(test_indices)
print(f"CIFAR-2: {N_TRAIN} train, {N_TEST} test")

# Preload all data into tensors for fast subset retraining
def preload(subset):
    xs, ys = [], []
    for x, y in subset:
        xs.append(x)
        ys.append(y)
    return torch.stack(xs), torch.tensor(ys)

X_train_all, y_train_all = preload(train_subset)
X_test_all, y_test_all = preload(test_subset)
print(f"X_train: {X_train_all.shape}, X_test: {X_test_all.shape}")

## 2. ResNet-9 Model

In [ ]:
def conv_bn(in_c, out_c, kernel_size=3, stride=1, padding=1):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size, stride, padding, bias=False),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
    )

class ResNet9(nn.Module):
    """Compact ResNet-9 for CIFAR (following DAWNBench / TRAK paper setup)."""
    def __init__(self, num_classes=2):
        super().__init__()
        self.prep = conv_bn(3, 64)
        self.layer1 = nn.Sequential(conv_bn(64, 128), nn.MaxPool2d(2))
        self.res1 = nn.Sequential(conv_bn(128, 128), conv_bn(128, 128))
        self.layer2 = nn.Sequential(conv_bn(128, 256), nn.MaxPool2d(2))
        self.layer3 = nn.Sequential(conv_bn(256, 512), nn.MaxPool2d(2))
        self.res2 = nn.Sequential(conv_bn(512, 512), conv_bn(512, 512))
        self.pool = nn.AdaptiveMaxPool2d(1)
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.prep(x)
        x = self.layer1(x)
        x = x + self.res1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = x + self.res2(x)

        x = self.pool(x).flatten(1)
        return self.classifier(x)

def train_resnet(model, X, y, lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE):
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, 
                                              steps_per_epoch=(len(X) // batch_size + 1), 
                                              epochs=epochs)
    loss_fn = nn.CrossEntropyLoss()
    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)
    for epoch in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            scheduler.step()
    return model

@torch.no_grad()
def get_accuracy(model, X, y, batch_size=512):
    model.eval()
    correct = 0
    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=batch_size)
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        correct += (model(xb).argmax(1) == yb).sum().item()
    return correct / len(X)

@torch.no_grad()
def get_per_sample_correct(model, X, y, batch_size=512):
    """Returns a binary array: 1 if model predicts correctly, 0 otherwise."""
    model.eval()
    results = []
    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=batch_size)
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        results.append((model(xb).argmax(1) == yb).cpu().numpy())
    return np.concatenate(results).astype(np.float64)

## 3. Train Full Model

In [ ]:
torch.manual_seed(SEED)
model = ResNet9(num_classes=2).to(DEVICE)
model = train_resnet(model, X_train_all, y_train_all)
full_acc = get_accuracy(model, X_test_all, y_test_all)
print(f"Full model test accuracy: {full_acc:.4f}")

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

## 4. TRAK Attribution (dattri)

In [ ]:
print("Computing TRAK attribution scores...")
t0 = time.perf_counter()

trak_success = False
trak_influence_matrix = None

try:
    from dattri.algorithm.trak import TRAKAttributor

    train_loader = DataLoader(
        TensorDataset(X_train_all, y_train_all), batch_size=256, shuffle=False
    )
    test_loader = DataLoader(
        TensorDataset(X_test_all, y_test_all), batch_size=256, shuffle=False
    )

    # Move model to eval for attribution
    model.eval()

    def loss_trak(params, data_target_pair):
        x, y = data_target_pair
        out = torch.func.functional_call(model, params, (x,))
        return nn.functional.cross_entropy(out, y, reduction="sum")

    attributor = TRAKAttributor(
        target_func=loss_trak,
        params=dict(model.named_parameters()),
        weight_list=[dict(model.named_parameters())],
        proj_dim=PROJ_DIM,
        device=DEVICE,
    )

    attributor.cache(train_loader)
    trak_scores = attributor.attribute(test_loader)  # (N_TEST, N_TRAIN)
    trak_influence_matrix = trak_scores.cpu().numpy()
    trak_success = True
    print(f"dattri TRAK succeeded! Shape: {trak_influence_matrix.shape}")

except Exception as e:
    print(f"dattri TRAK failed: {e}")
    print("Falling back to manual TRAK (random projection of per-sample gradients)...")

trak_time = time.perf_counter() - t0
print(f"TRAK step took {trak_time:.1f}s")

In [ ]:
# Fallback: manual TRAK if dattri failed
if not trak_success:
    print("Running manual TRAK (per-sample gradient projection)...")
    t0 = time.perf_counter()
    model.eval()
    
    # Get parameter count for projection
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Use only the last-layer gradients for tractability (common TRAK approximation)
    # Last layer: model.classifier (Linear 512->2) = 512*2 + 2 = 1026 params
    last_layer_dim = model.classifier.weight.numel() + model.classifier.bias.numel()
    print(f"Using last-layer gradients only: {last_layer_dim} params")
    
    proj_matrix = torch.randn(last_layer_dim, PROJ_DIM, device=DEVICE) / (PROJ_DIM ** 0.5)
    loss_fn = nn.CrossEntropyLoss()
    
    def get_last_layer_grad(x, y_label):
        model.zero_grad()
        out = model(x.unsqueeze(0).to(DEVICE))
        loss = loss_fn(out, y_label.unsqueeze(0).to(DEVICE))
        loss.backward()
        return torch.cat([model.classifier.weight.grad.flatten(),
                          model.classifier.bias.grad.flatten()])
    
    # Project training gradients
    print("  Projecting training gradients...")
    train_projs = torch.zeros(N_TRAIN, PROJ_DIM, device=DEVICE)
    for i in range(N_TRAIN):
        g = get_last_layer_grad(X_train_all[i], y_train_all[i])
        train_projs[i] = g @ proj_matrix
        if i % 1000 == 0:
            print(f"    Train {i}/{N_TRAIN}")
    
    # Project test gradients
    print("  Projecting test gradients...")
    test_projs = torch.zeros(N_TEST, PROJ_DIM, device=DEVICE)
    for i in range(N_TEST):
        g = get_last_layer_grad(X_test_all[i], y_test_all[i])
        test_projs[i] = g @ proj_matrix
        if i % 500 == 0:
            print(f"    Test {i}/{N_TEST}")
    
    trak_influence_matrix = (test_projs @ train_projs.T).cpu().numpy()
    trak_time = time.perf_counter() - t0
    print(f"Manual TRAK took {trak_time:.1f}s")
    print(f"Shape: {trak_influence_matrix.shape}")

## 5. Traceprop Attribution

Traceprop uses **batch-mean** gradients (constant memory). We compute the batch-mean gradient for each training sample's batch, project it, and use dot-product influence.

In [ ]:
print("Computing Traceprop attribution scores (batch-mean gradients)...")
t0 = time.perf_counter()

# Sparse JL projection matrix (matches traceprop.attribution.gradient_store)
rng = np.random.default_rng(42)
last_layer_dim = model.classifier.weight.numel() + model.classifier.bias.numel()
jl_matrix = rng.choice(
    [-1.0, 0.0, 1.0],
    size=(PROJ_DIM, last_layer_dim),
    p=[1/6, 2/3, 1/6],
).astype(np.float32) * np.sqrt(3.0 / PROJ_DIM)
jl_matrix_t = torch.from_numpy(jl_matrix).float().to(DEVICE)

loss_fn = nn.CrossEntropyLoss()
TP_BATCH = BATCH_SIZE

model.eval()
train_dataset = TensorDataset(X_train_all, y_train_all)
train_loader_seq = DataLoader(train_dataset, batch_size=TP_BATCH, shuffle=False)

tp_projected_grads = []

sample_idx = 0
for batch_i, (xb, yb) in enumerate(train_loader_seq):
    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
    model.zero_grad()
    out = model(xb)
    loss = loss_fn(out, yb)
    loss.backward()
    
    batch_grad = torch.cat([model.classifier.weight.grad.flatten(),
                            model.classifier.bias.grad.flatten()]).float()
    
    proj = jl_matrix_t @ batch_grad
    
    bs = xb.shape[0]
    for _ in range(bs):
        tp_projected_grads.append(proj)
    sample_idx += bs
    
    if batch_i % 20 == 0:
        print(f"  Batch {batch_i}, samples {sample_idx}/{N_TRAIN}")

tp_train_matrix = torch.stack(tp_projected_grads)

tp_test_projs = []
for i in range(N_TEST):
    x = X_test_all[i].unsqueeze(0).to(DEVICE)
    y = y_test_all[i].unsqueeze(0).to(DEVICE)
    model.zero_grad()
    out = model(x)
    loss = loss_fn(out, y)
    loss.backward()
    test_grad = torch.cat([model.classifier.weight.grad.flatten(),
                           model.classifier.bias.grad.flatten()]).float()
    tp_test_projs.append(jl_matrix_t @ test_grad)
    if i % 500 == 0:
        print(f"  Test {i}/{N_TEST}")

tp_test_matrix = torch.stack(tp_test_projs)

tp_influence_matrix = (tp_test_matrix @ tp_train_matrix.T).cpu().numpy()
tp_time = time.perf_counter() - t0
print(f"Traceprop attribution took {tp_time:.1f}s")
print(f"Shape: {tp_influence_matrix.shape}")

## 6. Ground-Truth Retraining (LDS)

In [ ]:
print(f"Retraining on {N_SUBSETS} random subsets for ground truth...")
t0 = time.perf_counter()

np.random.seed(SEED + 1000)

subset_masks = []
subset_correct = np.zeros((N_SUBSETS, N_TEST_LDS))

for s in range(N_SUBSETS):
    mask = np.random.rand(N_TRAIN) < SUBSET_RATIO
    subset_masks.append(mask)
    
    X_sub = X_train_all[mask]
    y_sub = y_train_all[mask]
    
    torch.manual_seed(s)
    m = ResNet9(num_classes=2).to(DEVICE)
    m = train_resnet(m, X_sub, y_sub)
    # Evaluate on first N_TEST_LDS test points only
    subset_correct[s] = get_per_sample_correct(m, X_test_all[:N_TEST_LDS], y_test_all[:N_TEST_LDS])
    
    acc = subset_correct[s].mean()
    print(f"  Subset {s}/{N_SUBSETS}: {mask.sum()} samples, acc={acc:.4f}")

subset_masks = np.array(subset_masks)
retrain_time = time.perf_counter() - t0
print(f"Retraining took {retrain_time:.1f}s ({retrain_time/60:.1f} min)")

## 7. Compute LDS

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

def compute_lds(influence_matrix, subset_masks, subset_correct, n_train, n_test):
    """Compute LDS: Spearman correlation between predicted and actual influence."""
    lds_scores = []
    for test_i in range(n_test):
        pred = influence_matrix[test_i]
        gt = np.array([
            spearmanr(subset_masks[:, j], subset_correct[:, test_i]).statistic
            for j in range(n_train)
        ])
        gt = np.nan_to_num(gt, 0.0)
        lds = spearmanr(pred, gt).statistic
        lds_scores.append(lds if not np.isnan(lds) else 0.0)
    return np.array(lds_scores)

print(f"Computing LDS for all methods (on {N_TEST_LDS} test points)...")

# Use only first N_TEST_LDS test points for LDS evaluation
lds_trak = compute_lds(trak_influence_matrix[:N_TEST_LDS], subset_masks, subset_correct, N_TRAIN, N_TEST_LDS)
print(f"TRAK LDS:       {lds_trak.mean():.4f} +/- {lds_trak.std():.4f}")

lds_tp = compute_lds(tp_influence_matrix[:N_TEST_LDS], subset_masks, subset_correct, N_TRAIN, N_TEST_LDS)
print(f"Traceprop LDS:  {lds_tp.mean():.4f} +/- {lds_tp.std():.4f}")

np.random.seed(SEED + 2000)
random_matrix = np.random.rand(N_TEST_LDS, N_TRAIN)
lds_rand = compute_lds(random_matrix, subset_masks, subset_correct, N_TRAIN, N_TEST_LDS)
print(f"Random LDS:     {lds_rand.mean():.4f} +/- {lds_rand.std():.4f}")

In [ ]:
print("\n" + "=" * 60)
print("CIFAR-2 / ResNet-9 — LDS Results")
print("=" * 60)
print(f"{'Method':<25} {'LDS (mean +/- std)':>20}")
print("-" * 50)
print(f"{'TRAK (per-sample)':<25} {lds_trak.mean():>8.4f} +/- {lds_trak.std():.4f}")
print(f"{'Traceprop (batch-mean)':<25} {lds_tp.mean():>8.4f} +/- {lds_tp.std():.4f}")
print(f"{'Random baseline':<25} {lds_rand.mean():>8.4f} +/- {lds_rand.std():.4f}")
print("=" * 60)

results = {
    "benchmark": "CIFAR-2/ResNet-9",
    "trak_lds_mean": round(float(lds_trak.mean()), 4),
    "trak_lds_std": round(float(lds_trak.std()), 4),
    "traceprop_lds_mean": round(float(lds_tp.mean()), 4),
    "traceprop_lds_std": round(float(lds_tp.std()), 4),
    "random_lds_mean": round(float(lds_rand.mean()), 4),
    "random_lds_std": round(float(lds_rand.std()), 4),
    "full_model_accuracy": round(full_acc, 4),
    "n_train": N_TRAIN,
    "n_test": N_TEST,
    "n_test_lds": N_TEST_LDS,
    "n_subsets": N_SUBSETS,
    "proj_dim": PROJ_DIM,
    "epochs": EPOCHS,
    "trak_time_s": round(trak_time, 1),
    "traceprop_time_s": round(tp_time, 1),
    "retrain_time_s": round(retrain_time, 1),
    "trak_method": "dattri" if trak_success else "manual_last_layer",
    "device": str(DEVICE),
}

with open("cifar2_resnet9_lds.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved to cifar2_resnet9_lds.json")
print(json.dumps(results, indent=2))